In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoConfig, AutoTokenizer
from pathlib import Path

device = 'cuda'
dtype = torch.bfloat16

model_id="Qwen/Qwen3-0.6B"
model_id: str = "yujiepan/qwen3-tiny-random"



In [ ]:

# load base model
model_config = AutoConfig.from_pretrained(
    model_id,
)


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=model_config,
    device_map=device,
    torch_dtype=dtype,
)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    use_fast=True,
)

`torch_dtype` is deprecated! Use `dtype` instead!


In [ ]:

messages_split = [
    {"role": "user", "content": "The capital of Mars is"},
]
inputs = tokenizer.apply_chat_template(messages_split, return_tensors="pt", padding='max_length', max_length=53, return_dict=True, return_attention_mask=True)

print("Input tokens:", tokenizer.batch_decode(inputs["input_ids"], skip_special_tokens=False))
inputs = {k: v.to(device) for k, v in inputs.items()}


# We want to test, forward then generate with past_key_values
model.eval()
with torch.no_grad():
    with torch.autocast(device_type=device, dtype=dtype):
        outputs = model.forward(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
        )


Input tokens: ['<|im_start|>user\nThe capital of Mars is<|im_end|>\n<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>']


In [ ]:
def kv_cache_shape(kv_cache):
    # [layers, batch, n_heads, seq_len, head_dim]
    return (len(kv_cache.layers),) + tuple(kv_cache.layers[0].values.shape)


In [ ]:
import copy

next_token = outputs.logits[:, -1:, :].log_softmax(dim=-1).argmax(dim=-1)
new_att_mask = torch.cat(
    [inputs["attention_mask"],
     torch.ones_like(next_token).long()],
    dim=-1,
)
kv_cache = copy.deepcopy(outputs.past_key_values)
cache_len = kv_cache.layers[0].values.shape[2]

print("Original input shape:", inputs["input_ids"].shape)
print("Next token shape:", next_token.shape)
print("New attention mask shape:", new_att_mask.shape)
print("KV cache shapes  [layers, batch, n_heads, seq_len, head_dim] :", kv_cache_shape(outputs.past_key_values))
with torch.autocast(device_type=device, dtype=dtype):
    with torch.no_grad():
        outputs2 = model.generate(
            input_ids=next_token,
            attention_mask=new_att_mask,
            past_key_values=kv_cache,
            cache_position=torch.ones_like(next_token) * cache_len,
        )

print(tokenizer.batch_decode(outputs2.cpu())[0])

Original input shape: torch.Size([1, 53])
Next token shape: torch.Size([1, 1])
New attention mask shape: torch.Size([1, 54])
KV cache shapes  [layers, batch, n_heads, seq_len, head_dim] : (2, 1, 1, 53, 32)


RuntimeError: expand(CUDABFloat16Type{[1, 1, 1, 54, 1]}, size=[1, 2, 1, 54]): the number of sizes provided (4) must be greater or equal to the number of dimensions in the tensor (5)

In [ ]:

kv_cache_shape(kv_cache), kv_cache_shape(outputs.past_key_values)

((2, 1, 1, 30, 32), (2, 1, 1, 10, 32))

In [ ]:
[l.values.shape for l in kv_cache.layers]

[torch.Size([1, 1, 30, 32]), torch.Size([1, 1, 30, 32])]

In [ ]:
[l.values.shape for l in outputs.past_key_values.layers]

[torch.Size([1, 1, 10, 32]), torch.Size([1, 1, 10, 32])]